# Automated Performace Testing of Sighting Assistant Tool. 

### How to use this notebook

**1.** Input your HSD ID or Query ID.  
**2.** Click on "Run" > "Run all cells" in the menu on top of this notebook.  
**3.** Review content of each section of the Sighting Assistant tool.      
**4.** At the last cell of this notebook, Give Feedback by rating it on a scale of 1–10 and providing feedback text.    
**5.** Click on the "Save Assessment Results" button at the end.    
**6.** Find the Sighting Assistant tool's output in `{sighting_home_directory}/accuracy_test/runs/`.    
**7.** Find the user's feedback and performance/usage/duration stats in `{sighting_home_directory}/accuracy_test/data/`.  
**8.** Click on Edit > Clear Output of All Cells for a Clean view for next iteration .  

### USER CONFIGURATION - CHOOSE ONE OF THE OPTIONS BELOW (Uncomment acc. to requirement)


#### Option 1: Use a Query ID to retrieve multiple HSD IDs (limited to first 5)

In [7]:

#USER_INPUT_TYPE = "query"  # Set to "query" or "hsd_id"
#QUERY_ID = 15017678515  # ← Set your query ID here



#### Option 2: Use a single HSD ID

In [8]:
USER_INPUT_TYPE = "hsd_id"  # Set to "query" or "hsd_id"  
SINGLE_HSD_ID = "14026454583"  # ← Set your single HSD ID here

In [9]:
%pip install ipywidgets 


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


----

## Run All cells by Clicking on the Menu on top, "Run" > "Run All Cells"

#### HTML related code

In [10]:

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import re
import json
from datetime import datetime
import os
import chardet
import time
import traceback
import subprocess

# Enhanced MakeSlider class with content display
class MakeSliderWithContent:
    def __init__(self, name, content="", initial=5, min_val=0, max_val=10, step=1):
        self.name = name
        self.content = content
        
        # Create slider
        self.slider = widgets.IntSlider(
            value=initial,
            min=min_val,
            max=max_val,
            step=step,
            description=f'{name}:',
            continuous_update=True,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='100%')
        )

        # Create remark text area
        self.remark = widgets.Textarea(
            value='',
            placeholder='Enter detailed feedback and reason for this rating...',
            description='Feedback:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='100%', height='80px')
        )

        # Create content display area
        self.content_display = widgets.HTML(
            value=self._format_content_html(),
            layout=widgets.Layout(
                width='100%', 
                height='400px', 
                overflow='auto',
                border='1px solid #ccc',
                padding='10px',
                margin='5px 0'
            )
        )

        # Create collapsible content section
        self.content_accordion = widgets.Accordion(
            children=[self.content_display],
            titles=[f'📄 {name} - Full Content']
        )
        self.content_accordion.selected_index = None  # Start collapsed

        self.output = widgets.Output()

        # Initialize data
        self.value = self.slider.value
        self.remark_text = self.remark.value

        # Set up observers
        self.slider.observe(self._on_slider_change, names='value')
        self.remark.observe(self._on_remark_change, names='value')

        # Create the main widget layout
        main_widget = widgets.VBox([
            widgets.HTML(f"<h3 style='color: #2E86AB; margin: 10px 0;'>🎯 {name}</h3>"),
            self.slider,
            self.remark,
            self.content_accordion,
            self.output
        ], layout=widgets.Layout(
            border='2px solid #f0f0f0',
            margin='10px 0',
            padding='15px',
            border_radius='8px'
        ))

        # Display the widget
        display(main_widget)





    
    def _format_content_html(self):
        
        """Format content as HTML for better display"""
        if not self.content:
            return "<p><em>No content available for this section</em></p>"
        
        # Simple approach - preserve exact formatting
        import html
        escaped_content = html.escape(self.content)
        
        # Convert basic markdown to HTML for headers and bold text
        escaped_content = re.sub(r'\*\*(.*?)\*\*', r'<strong>\1</strong>', escaped_content)
        escaped_content = re.sub(r'### (.*?)\n', r'<h4 style="color: #2E86AB; margin-top: 15px;">\1</h4>\n', escaped_content)
        escaped_content = re.sub(r'## (.*?)\n', r'<h3 style="color: #A23B72; margin-top: 20px;">\1</h3>\n', escaped_content)
        
        # Style the content with monospace font to preserve table alignment
        styled_content = f"""
        <div style="font-family: 'Courier New', monospace; 
                    font-size: 11px; 
                    line-height: 1.3; 
                    color: #333;
                    background-color: #f8f9fa;
                    padding: 15px;
                    border-radius: 5px;
                    max-height: 380px;
                    overflow-y: auto;
                    white-space: pre-wrap;
                    border: 1px solid #dee2e6;">
            {escaped_content}
        </div>
        """
        
        return styled_content
      
    
    
    def _on_slider_change(self, change):
        self.value = change['new']
        with self.output:
            self.output.clear_output()
            print(f"✅ {self.name} rating updated to {self.value}/10")

    def _on_remark_change(self, change):
        self.remark_text = change['new']
        with self.output:
            self.output.clear_output()
            if self.remark_text:
                print(f"📝 Feedback updated for {self.name}")

    def get_data(self):
        return {
            'name': self.name,
            'value': self.value,
            'remark': self.remark_text,
            'content_length': len(self.content)
        }

# DataFrame structure (unchanged)
columns_list = [
    ('HSD ID', ''),
    ('Content Extraction & Summary', 'Attachment Info File Path'),
    ('Content Extraction & Summary', 'Title'),
    ('Content Extraction & Summary', 'Main issue'),
    ('Content Extraction & Summary', 'Description'),
    ('Content Extraction & Summary', 'ETL Attachment Info: Names'),
    ('Content Extraction & Summary', 'ETL Attachment Info: Types'),
    ('Content Extraction & Summary', 'ETL Attachment Info: Notes'),
    ('Content Extraction & Summary', 'Driver Info: Build Types'),
    ('Content Extraction & Summary', 'Driver Info: Versions'),
    ('Content Extraction & Summary', 'Driver Info: Build Dates'),
    ('Content Extraction & Summary', 'Driver Info: ETL File Names'),
    ('Content Extraction & Summary', 'Driver Info: Age Warning'),
    ('Content Extraction & Summary', 'Pipe Underrun Check'),
    ('Content Extraction & Summary', 'Regression Analysis'),
    ('Content Extraction & Summary', 'RVP Reproducibility Status'),
    ('Content Extraction & Summary', 'Reproduction Instructions'),
    ('Content Extraction & Summary', 'System Configuration'),
    ('Issue Classification', 'Category'),
    ('Issue Classification', 'Agent Output'),
    ('Attachment Verification', 'Expected Attachments'),
    ('Attachment Verification', 'Present Attachments'),
    ('Attachment Verification', 'Missing Attachments'),
    ('Attachment Verification', 'Unexpected Attachments'),
    ('Attachment Verification', 'Validity Status'),
    ('Attachment Verification', 'Validity Reason'),
    ('Attachment Verification', 'Output Statement'),
    ('Suggestions as per DFD checklist', 'Mandatory DFD Analysis Output'),
    ('Suggestions as per DFD checklist', 'Compliance Status'),
    ('Suggestions as per DFD checklist', 'BKM Tool Used'),
    ('Suggestions as per DFD checklist', 'BKM Output'),
    ('Suggestions as per DFD checklist', 'BKM Suggestions'),
    ('Triage and Troubleshooting Review', 'Comments Analysis'),
    ('Triage and Troubleshooting Review', 'Troubleshooting Performed'),
    ('Triage and Troubleshooting Review', 'Successful Mitigations'),
    ('Triage and Troubleshooting Review', 'Failed Mitigations'),
    ('Triage and Troubleshooting Review', 'Pending Actions'),
    ('Triage and Troubleshooting Review', 'Investigation Status'),
    ('Triage and Troubleshooting Review', 'Recommended Next Steps'),
    ('Executive Summary and Recommendations', 'Technical Summary'),
    ('Executive Summary and Recommendations', 'Actionable Recommendations'),
    ('Executive Summary and Recommendations', 'Priority Level'),
    ('Executive Summary and Recommendations', 'Escalation Path'),
    ('Executive Summary and Recommendations', 'Missing Information'),
    ('Tool Invocation Log', 'Sighting Attachments Invoked'),
    ('Tool Invocation Log', 'Sighting Get Category Invoked'),
    ('Tool Invocation Log', 'Mandatory DFD Analyzer Invoked'),
    ('Tool Invocation Log', 'Category Specific BKM Tool Invoked'),
    ('Tool Invocation Log', 'Tools Summary')
]

columns = pd.MultiIndex.from_tuples(columns_list)
df = pd.DataFrame(columns=columns)

def detect_file_encoding(file_path):
    """Detect the encoding of a file"""
    try:
        with open(file_path, 'rb') as f:
            raw_data = f.read(10000)  # Read first 10KB to detect encoding
        result = chardet.detect(raw_data)
        return result['encoding']
    except:
        return 'utf-8'

def read_file_with_fallback_encoding(file_path):
    """Read file with multiple encoding attempts"""
    encodings_to_try = [
        'utf-8',
        'utf-8-sig',  # UTF-8 with BOM
        'cp1252',     # Windows-1252
        'iso-8859-1', # Latin-1
        'ascii',
        'utf-16',
        'utf-16le',
        'utf-16be'
    ]
    
    # First, try to detect encoding
    detected_encoding = detect_file_encoding(file_path)
    if detected_encoding:
        encodings_to_try.insert(0, detected_encoding)
    
    print(f"Attempting to read file: {file_path}")
    
    for encoding in encodings_to_try:
        try:
            print(f"Trying encoding: {encoding}")
            with open(file_path, 'r', encoding=encoding, errors='replace') as f:
                content = f.read()
            print(f"✅ Successfully read file with encoding: {encoding}")
            print(f"File size: {len(content)} characters")
            return content
        except UnicodeDecodeError as e:
            print(f"❌ Failed with {encoding}: {e}")
            continue
        except Exception as e:
            print(f"❌ Error with {encoding}: {e}")
            continue
    
    # Last resort: read as binary and decode with errors='replace'
    try:
        print("Last resort: Reading as binary with error replacement")
        with open(file_path, 'rb') as f:
            raw_content = f.read()
        content = raw_content.decode('utf-8', errors='replace')
        print(f"✅ Successfully read file as binary with error replacement")
        return content
    except Exception as e:
        print(f"❌ Final attempt failed: {e}")
        return None






class GNAIOutputParser:
    def __init__(self):
        self.parsed_data = {}
        self.section_contents = {}  # Store raw section contents
        self.usage_statistics = {}  # Add this line
        



    # ADD THIS NEW METHOD HERE:
    def _extract_hsd_id(self, content):
        """Extract HSD ID from multiple possible locations in the content"""
        
        print("🔍 Searching for HSD ID in content...")
        
        # Method 1: Look in the structured content under Basic Information
        structured_id_patterns = [
            r'\*\*ID:\*\*\s*(\d+)',  # **ID:** 15018275324
            r'### 1\.1 Basic Information\n.*?\*\*ID:\*\*\s*(\d+)',  # In Basic Information section
            r'ID:\s*(\d+)',  # Simple ID: pattern
            r'HSD.*?ID.*?(\d{8,})',  # HSD ID with 8+ digits
        ]
        
        for i, pattern in enumerate(structured_id_patterns, 1):
            print(f"   Trying pattern {i}: {pattern}")
            match = re.search(pattern, content, re.DOTALL | re.IGNORECASE)
            if match:
                hsd_id = match.group(1)
                print(f"✅ Found HSD ID using pattern {i}: {hsd_id}")
                return hsd_id
            else:
                print(f"   ❌ Pattern {i} failed")
        
        # Method 2: Look in the command line (original method as fallback)
        command_patterns = [
            r"'Give summary of HSD id (\d+)'",
            r"Give summary of HSD id (\d+)",
            r"HSD id (\d+)",
            r"hsd[_\s]*id[:\s]*(\d+)",
        ]
        
        print("🔍 Trying command line patterns...")
        for i, pattern in enumerate(command_patterns, 1):
            print(f"   Trying command pattern {i}: {pattern}")
            match = re.search(pattern, content, re.IGNORECASE)
            if match:
                hsd_id = match.group(1)
                print(f"✅ Found HSD ID in command using pattern {i}: {hsd_id}")
                return hsd_id
            else:
                print(f"   ❌ Command pattern {i} failed")
        
        # Method 3: Look for the specific content structure you showed
        content_structure_patterns = [
            r'## 1\. \*\*Content Extraction & Summary\*\*.*?### 1\.1 Basic Information.*?\*\*ID:\*\*\s*(\d+)',
            r'Basic Information.*?\*\*ID:\*\*\s*(\d+)',
        ]
        
        print("🔍 Trying content structure patterns...")
        for i, pattern in enumerate(content_structure_patterns, 1):
            print(f"   Trying structure pattern {i}")
            match = re.search(pattern, content, re.DOTALL | re.IGNORECASE)
            if match:
                hsd_id = match.group(1)
                print(f"✅ Found HSD ID in content structure using pattern {i}: {hsd_id}")
                return hsd_id
            else:
                print(f"   ❌ Structure pattern {i} failed")
        
        # Method 4: Look for any long numeric ID that might be the HSD ID
        print("🔍 Looking for any long numeric patterns...")
        numeric_id_patterns = [
            r'\b(\d{10,})\b',  # Any number with 10+ digits
            r'\b(\d{8,})\b',   # Any number with 8+ digits
        ]
        
        for i, pattern in enumerate(numeric_id_patterns, 1):
            matches = re.findall(pattern, content)
            if matches:
                # Take the first long number found
                hsd_id = matches[0]
                print(f"⚠️ Found potential HSD ID using numeric pattern {i}: {hsd_id}")
                print(f"   All matches found: {matches[:5]}")  # Show first 5 matches
                return hsd_id
        
        # Debug: Show a sample of the content around where we expect to find the ID
        print("🔍 DEBUG: Looking for content around 'Basic Information'...")
        basic_info_match = re.search(r'Basic Information.*?(\n.*?){0,10}', content, re.DOTALL | re.IGNORECASE)
        if basic_info_match:
            print(f"   Found Basic Information section: {basic_info_match.group(0)[:500]}...")
        else:
            print("   ❌ No 'Basic Information' section found")
        
        # Debug: Show content around Content Extraction
        print("🔍 DEBUG: Looking for content around 'Content Extraction'...")
        content_extract_match = re.search(r'Content Extraction.*?(\n.*?){0,15}', content, re.DOTALL | re.IGNORECASE)
        if content_extract_match:
            print(f"   Found Content Extraction section: {content_extract_match.group(0)[:500]}...")
        else:
            print("   ❌ No 'Content Extraction' section found")
        
        print("❌ Could not extract HSD ID from any location")
        return "Unknown"



    def debug_section_extraction(self, content):
        """Debug function to see what's being extracted"""
        structured_start = content.find('## 1. **Content Extraction & Summary**')
        if structured_start != -1:
            structured_content = content[structured_start:]
            
            # Find section 1
            section1_match = re.search(r'## 1\. \*\*Content Extraction & Summary\*\*(.*?)(?=## 2\. \*\*Issue Classification\*\*|$)', structured_content, re.DOTALL)
            if section1_match:
                section1_content = section1_match.group(1).strip()
                print(f"DEBUG: Section 1 content length: {len(section1_content)}")
                print(f"DEBUG: Section 1 preview: {section1_content[:500]}...")
                
                # Check if ETL table is complete
                if '### 1.3 ETL Attachment Analysis' in section1_content:
                    etl_start = section1_content.find('### 1.3 ETL Attachment Analysis')
                    etl_section = section1_content[etl_start:]
                    print(f"DEBUG: ETL section: {etl_section[:500]}...")
            else:
                print("DEBUG: Could not find Section 1 content")
        else:
            print("DEBUG: Could not find structured content start")



    def parse_output_file(self, file_path):
        """Parse the GNAI output file and extract structured information"""
        
        # Check if file exists
        if not os.path.exists(file_path):
            print(f"❌ File not found: {file_path}")
            return None
        
        # Read file with encoding fallback
        content = read_file_with_fallback_encoding(file_path)
        if content is None:
            print("❌ Could not read the file with any encoding")
            return None
        
        print(f"✅ File loaded successfully. Content length: {len(content)} characters")
        
        # Clean up content - remove problematic characters
        content = self._clean_content(content)

        # Extract HSD ID - UPDATED to use the new method
        hsd_id = self._extract_hsd_id(content)
        print(f"📋 Extracted HSD ID: {hsd_id}")       


        # Add this line after content = self._clean_content(content)
        self.debug_section_extraction(content)


        # Extract HSD ID from the command
        hsd_id_match = re.search(r"'Give summary of HSD id (\d+)'", content)
        hsd_id = hsd_id_match.group(1) if hsd_id_match else "Unknown"
        print(f"📋 Extracted HSD ID: {hsd_id}")
        
        # Parse attachment info from stdout
        attachment_info = self._parse_attachment_info(content)
        print(f"📎 Found {len(attachment_info.get('attachment_list', []))} attachments")
        
        # Parse driver info from stdout
        driver_info = self._parse_driver_info(content)
        print(f"🔧 Found {len(driver_info)} driver configurations")
        
        # Parse pipe underrun info
        pipe_underrun_info = self._parse_pipe_underrun_info(content)
        print(f"🔍 Pipe underrun detected: {pipe_underrun_info.get('detected', False)}")
        
        # Parse the final structured output (sections 1-6) and store raw content
        structured_output = self._parse_structured_output(content)
        print(f"📊 Parsed {len(structured_output)} structured sections")
        
        # Parse tool invocation log
        tool_log = self._parse_tool_invocations(content)
        print(f"🛠️ Tool invocations detected: {sum(tool_log.values())}")
        
        # Parse usage statistics (add this after other parsing)
        usage_stats = self._parse_usage_statistics(content)
        print(f"💰 Usage statistics extracted: ${usage_stats.get('total_cost', 'N/A')} cost")

        # Parse processing duration (add this after usage statistics parsing)
        duration_info = self._parse_processing_duration(content)
        if 'duration_seconds' in duration_info:
            print(f"⏱️ Processing duration: {duration_info['duration_formatted']} ({duration_info['duration_seconds']} seconds)")
        
        return {
            'hsd_id': hsd_id,
            'attachment_info': attachment_info,
            'driver_info': driver_info,
            'pipe_underrun_info': pipe_underrun_info,
            'structured_output': structured_output,
            'tool_log': tool_log,
            'section_contents': self.section_contents,
            'usage_statistics': usage_stats,
            'processing_duration': duration_info,  # Add this line
            'raw_content': content
        }





    def _parse_usage_statistics(self, content):
        """Parse usage statistics from the debug output"""
        usage_stats = {}
        
        # Look for the usage statistics section
        usage_pattern = r'usage:\s*\n(.*?)(?=\n\n|\[Process exited|$)'
        usage_match = re.search(usage_pattern, content, re.DOTALL)
        
        if usage_match:
            usage_text = usage_match.group(1)
            
            # Parse cache statistics
            cache_section = re.search(r'cache:\s*\n(.*?)(?=\n\s*completion_tokens|$)', usage_text, re.DOTALL)
            if cache_section:
                cache_text = cache_section.group(1)
                usage_stats['cache'] = {}
                
                # Extract cache metrics
                cache_hits = re.search(r'cache_hits:\s*([\d.]+)', cache_text)
                hit_ratio = re.search(r'hit_ratio:\s*([\d.]+)', cache_text)
                misses = re.search(r'misses:\s*([\d.]+)', cache_text)
                read_tokens = re.search(r'read_tokens:\s*([\d.]+)', cache_text)
                write_tokens = re.search(r'write_tokens:\s*([\d.]+)', cache_text)
                
                if cache_hits:
                    usage_stats['cache']['cache_hits'] = float(cache_hits.group(1))
                if hit_ratio:
                    usage_stats['cache']['hit_ratio'] = float(hit_ratio.group(1))
                if misses:
                    usage_stats['cache']['misses'] = float(misses.group(1))
                if read_tokens:
                    usage_stats['cache']['read_tokens'] = float(read_tokens.group(1))
                if write_tokens:
                    usage_stats['cache']['write_tokens'] = float(write_tokens.group(1))
            
            # Parse other usage metrics
            completion_tokens = re.search(r'completion_tokens:\s*([\d.]+)', usage_text)
            prompt_tokens = re.search(r'prompt_tokens:\s*([\d.]+)', usage_text)
            successful_requests = re.search(r'successful_requests:\s*([\d.]+)', usage_text)
            total_cost = re.search(r'total_cost:\s*([\d.]+)', usage_text)
            total_tokens = re.search(r'total_tokens:\s*([\d.]+)', usage_text)
            
            if completion_tokens:
                usage_stats['completion_tokens'] = float(completion_tokens.group(1))
            if prompt_tokens:
                usage_stats['prompt_tokens'] = float(prompt_tokens.group(1))
            if successful_requests:
                usage_stats['successful_requests'] = float(successful_requests.group(1))
            if total_cost:
                usage_stats['total_cost'] = float(total_cost.group(1))
            if total_tokens:
                usage_stats['total_tokens'] = float(total_tokens.group(1))
        
        return usage_stats
    
    def _clean_content(self, content):
        """Clean problematic characters from content"""
        # Replace common problematic characters
        replacements = {
            'ΓÇô': '-',  # En dash
            'ΓÇÖ': "'",  # Right single quotation mark
            'ΓÇ£': '"',  # Left double quotation mark
            'ΓÇ¥': '"',  # Right double quotation mark
            'Γû╢∩╕Å': '...',  # Ellipsis or similar
            '≡ƒÆ¼': '🔧',  # Tool emoji replacement
            'ΓåÆ': '→',  # Arrow replacement
        }
        
        for old, new in replacements.items():
            content = content.replace(old, new)
        
        return content
    
    def _parse_structured_output(self, content):
        """Parse the structured output sections 1-6 and store raw content"""
        sections = {}
        
        # Find the start of structured output (after all the debug logs)
        structured_start = content.find('## 1. **Content Extraction & Summary**')
        if structured_start == -1:
            print("⚠️ Warning: Could not find structured output sections")
            return sections
        
        structured_content = content[structured_start:]
        
        # Updated section patterns - more greedy matching to capture full content
        section_patterns = {
            'content_extraction': r'## 1\. \*\*Content Extraction & Summary\*\*(.*?)(?=## 2\. \*\*Issue Classification\*\*|$)',
            'issue_classification': r'## 2\. \*\*Issue Classification\*\*(.*?)(?=## 3\. \*\*Attachment Verification\*\*|$)',
            'attachment_verification': r'## 3\. \*\*Attachment Verification\*\*(.*?)(?=## 4\. \*\*Suggestions as per DFD Checklist\*\*|$)',
            'dfd_checklist': r'## 4\. \*\*Suggestions as per DFD Checklist\*\*(.*?)(?=## 5\. \*\*Triage and Troubleshooting Review\*\*|$)',
            'triage_review': r'## 5\. \*\*Triage and Troubleshooting Review\*\*(.*?)(?=## 6\. \*\*Executive Summary & Recommendations\*\*|$)',
            'executive_summary': r'## 6\. \*\*Executive Summary & Recommendations\*\*(.*?)(?=## 7\.|---|\*\*End of Summary|$)'
        }
        
        for section_name, pattern in section_patterns.items():
            match = re.search(pattern, structured_content, re.DOTALL)
            if match:
                raw_content = match.group(1).strip()
                self.section_contents[section_name] = raw_content  # Store raw content
                sections[section_name] = self._parse_section_content(raw_content, section_name)
                print(f"✅ Parsed section: {section_name} (length: {len(raw_content)} chars)")
            else:
                print(f"⚠️ Warning: Could not find section: {section_name}")
                self.section_contents[section_name] = "Section not found in output"
        
        return sections



    # ... (rest of the parsing methods remain the same as in previous version)
    def _parse_attachment_info(self, content):
        """Parse attachment information from stdout"""
        attachments = []
        
        # Extract attachment list
        attachment_pattern = r'Attachment no\.(\d+) : (.+?)(?=\n|$)'
        matches = re.findall(attachment_pattern, content)
        
        for num, name in matches:
            attachments.append({'number': num, 'name': name.strip()})
        
        # Extract ETL type analysis
        etl_analysis = {}
        # More flexible pattern for ETL analysis
        etl_sections = re.findall(r'Attachment name : (.+?)\n.*?Attachment info.*?\[(.+?)\]', content, re.DOTALL)
        
        for attachment_name, info in etl_sections:
            etl_analysis[attachment_name.strip()] = info
        
        return {
            'attachment_list': attachments,
            'etl_analysis': etl_analysis
        }
    
    def _parse_driver_info(self, content):
        """Parse driver information from stdout"""
        drivers = []
        
        # Extract from JSON structure in stdout
        json_match = re.search(r'"driver_info": \[(.*?)\]', content, re.DOTALL)
        if json_match:
            try:
                # Extract individual driver objects
                driver_objects = re.findall(r'\{[^}]+\}', json_match.group(1))
                for driver_obj in driver_objects:
                    # Extract fields from each driver object
                    build_type_match = re.search(r'"build_type": "([^"]+)"', driver_obj)
                    version_match = re.search(r'"version": "([^"]+)"', driver_obj)
                    build_date_match = re.search(r'"build_date": "([^"]+)"', driver_obj)
                    file_match = re.search(r'"file": "([^"]+)"', driver_obj)
                    
                    if all([build_type_match, version_match, build_date_match, file_match]):
                        drivers.append({
                            'build_type': build_type_match.group(1),
                            'version': version_match.group(1),
                            'build_date': build_date_match.group(1),
                            'files': [file_match.group(1)]
                        })
            except Exception as e:
                print(f"⚠️ Error parsing driver JSON: {e}")
        
        # Fallback: Extract from formatted output
        if not drivers:
            driver_pattern = r'Driver Build Type: (.+?)\nDriver Version:\s+(.+?)\nDriver Build Date: (.+?)\n.*?Files:\s+(\d+) file\(s\)\n((?:\s+-\s.+\n)*)'
            matches = re.findall(driver_pattern, content, re.DOTALL)
            
            for build_type, version, build_date, file_count, files_section in matches:
                files = re.findall(r'\s+-\s(.+)', files_section)
                drivers.append({
                    'build_type': build_type.strip(),
                    'version': version.strip(),
                    'build_date': build_date.strip(),
                    'file_count': int(file_count),
                    'files': [f.strip() for f in files]
                })
        
        return drivers
    
    def _parse_pipe_underrun_info(self, content):
        """Parse pipe underrun information"""
        # Check JSON structure first
        json_match = re.search(r'"pipe_underrun_detected": (true|false)', content)
        if json_match:
            detected = json_match.group(1) == 'true'
            if detected:
                # Extract files with pipe underrun
                files_match = re.search(r'"pipe_underrun_files": \[(.*?)\]', content, re.DOTALL)
                if files_match:
                    files = re.findall(r'"attachment": "([^"]+)".*?"file": "([^"]+)"', files_match.group(1))
                    return {
                        'detected': True,
                        'count': len(files),
                        'files': files
                    }
            return {'detected': False, 'count': 0, 'files': []}
        
        # Fallback to text parsing
        pipe_underrun_match = re.search(r'Found pipe underrun in (\d+) ETL file\(s\):', content)
        if pipe_underrun_match:
            count = int(pipe_underrun_match.group(1))
            underrun_files = re.findall(r'Attachment: (.+?)\nETL File:\s+(.+?)\n', content)
            return {
                'detected': True,
                'count': count,
                'files': underrun_files
            }
        else:
            return {'detected': False, 'count': 0, 'files': []}
    
    def _parse_section_content(self, content, section_name):
        """Parse individual section content"""
        if section_name == 'content_extraction':
            return self._parse_content_extraction_section(content)
        elif section_name == 'issue_classification':
            return self._parse_issue_classification_section(content)
        elif section_name == 'attachment_verification':
            return self._parse_attachment_verification_section(content)
        elif section_name == 'dfd_checklist':
            return self._parse_dfd_checklist_section(content)
        elif section_name == 'triage_review':
            return self._parse_triage_review_section(content)
        elif section_name == 'executive_summary':
            return self._parse_executive_summary_section(content)
        
        return content.strip()


    def _parse_content_extraction_section(self, content):
        """Parse Content Extraction & Summary section"""
        data = {}
        
        # Extract ID, title and main issue from Basic Information section
        basic_info_pattern = r'### 1\.1 Basic Information\n(.*?)(?=### 1\.2|$)'
        basic_info_match = re.search(basic_info_pattern, content, re.DOTALL)
        
        if basic_info_match:
            basic_info_text = basic_info_match.group(1)
            
            # Extract ID
            id_match = re.search(r'\*\*ID:\*\*\s*(\d+)', basic_info_text)
            if id_match:
                data['id'] = id_match.group(1)
            
            # Extract title
            title_match = re.search(r'\*\*Title:\*\*\s*(.+?)(?=\n\*\*|\n\n|$)', basic_info_text, re.DOTALL)
            if title_match:
                data['title'] = title_match.group(1).strip()
            
            # Extract main issue
            main_issue_match = re.search(r'\*\*Main Issue:\*\*\s*(.+?)(?=\n\*\*|\n\n|$)', basic_info_text, re.DOTALL)
            if main_issue_match:
                data['main_issue'] = main_issue_match.group(1).strip()
        
        # Fallback patterns if Basic Information section not found
        if 'title' not in data:
            title_match = re.search(r'\*\*Title:\*\* (.+?)(?=\n)', content)
            data['title'] = title_match.group(1).strip() if title_match else ""
        
        if 'main_issue' not in data:
            main_issue_match = re.search(r'\*\*Main Issue:\*\* (.+?)(?=\n)', content)
            data['main_issue'] = main_issue_match.group(1).strip() if main_issue_match else ""
        
        # Extract description - look for various patterns
        desc_patterns = [
            r'### 1\.2 Detailed Problem Description and Context\n.*?\*\*Description:\*\*(.*?)(?=### 1\.3|$)',
            r'### 1\.2\) Detailed Problem Description and Context\n(.*?)(?=### 1\.3|$)',
            r'- \*\*Description:\*\*(.*?)(?=### 1\.3|$)'
        ]
        
        for pattern in desc_patterns:
            desc_match = re.search(pattern, content, re.DOTALL)
            if desc_match:
                data['description'] = desc_match.group(1).strip()
                break
        
        if 'description' not in data:
            data['description'] = ""
        
        # Extract ETL Attachment Analysis section as raw text
        etl_pattern = r'### 1\.3 ETL Attachment Analysis\n(.*?)(?=### 1\.4|$)'
        etl_match = re.search(etl_pattern, content, re.DOTALL)
        if etl_match:
            data['etl_attachments_raw'] = etl_match.group(1).strip()
        else:
            data['etl_attachments_raw'] = "No ETL attachment information found"
        
        # Keep empty for compatibility
        data['etl_attachments'] = []
        
        # Extract reproduction instructions
        repro_patterns = [
            r'### 1\.8 Step-by-Step Reproduction Instructions\n(.*?)(?=### 1\.9|---|\*\*|$)',
            r'### 1\.8\) Step-by-Step Reproduction Instructions\n(.*?)(?=### 1\.9|---|\*\*|$)'
        ]
        
        for pattern in repro_patterns:
            repro_match = re.search(pattern, content, re.DOTALL)
            if repro_match:
                repro_text = repro_match.group(1).strip()
                repro_steps = re.findall(r'(?:^|\n)\s*\d+\.\s*(.+?)(?=\n\s*\d+\.|$)', repro_text, re.DOTALL | re.MULTILINE)
                data['reproduction_instructions'] = [step.strip() for step in repro_steps]
                break
        
        if 'reproduction_instructions' not in data:
            data['reproduction_instructions'] = []
        
        # Extract system configuration
        sys_config_patterns = [
            r'### 1\.9 System Configuration and Environment Details\n(.*?)(?=---|## 2\.|$)',
            r'### 1\.9\) System Configuration and Environment Details\n(.*?)(?=---|## 2\.|$)'
        ]
        
        for pattern in sys_config_patterns:
            sys_config_match = re.search(pattern, content, re.DOTALL)
            if sys_config_match:
                data['system_configuration'] = sys_config_match.group(1).strip()
                break
        
        if 'system_configuration' not in data:
            data['system_configuration'] = ""
        
        return data




    def _parse_dfd_checklist_section(self, content):
        
        
        """Parse DFD Checklist section"""
        data = {}
        
        # Extract Mandatory DFD Checklist Compliance section as raw text
        checklist_pattern = r'### 4\.1 Mandatory DFD Checklist Compliance\n(.*?)(?=### 4\.2|### |---|\*\*|$)'
        checklist_match = re.search(checklist_pattern, content, re.DOTALL)
        
        if checklist_match:
            data['mandatory_checklist_raw'] = checklist_match.group(1).strip()
        else:
            data['mandatory_checklist_raw'] = "No mandatory checklist found"
        
        # Keep empty for compatibility
        data['mandatory_checklist'] = []
        
        return data

    
    
    def _parse_issue_classification_section(self, content):
        """Parse Issue Classification section"""
        category_match = re.search(r'\*\*Category:\*\* (.+?)(?=\n)', content)
        reason_patterns = [
            r'- Reason: (.+?)(?=\n|$)',
            r'\*\*Reasoning:\*\* (.+?)(?=\n|$)',
            r'Reason: (.+?)(?=\n|$)'
        ]
        
        reasoning = ""
        for pattern in reason_patterns:
            reason_match = re.search(pattern, content)
            if reason_match:
                reasoning = reason_match.group(1).strip()
                break
        
        return {
            'category': category_match.group(1).strip() if category_match else "",
            'reasoning': reasoning
        }
    
    def _parse_attachment_verification_section(self, content):
        """Parse Attachment Verification section"""
        data = {}
        
        # Extract validity status
        validity_patterns = [
            r'\*\*Valid article as per .+?: (.+?)\*\*',
            r'Valid article as per .+?: (.+?)(?=\n|$)'
        ]
        
        for pattern in validity_patterns:
            validity_match = re.search(pattern, content)
            if validity_match:
                data['validity_status'] = validity_match.group(1).strip()
                break
        
        if 'validity_status' not in data:
            data['validity_status'] = ""
        
        return data
    
   
    
    def _parse_triage_review_section(self, content):
        """Parse Triage and Troubleshooting Review section"""
        data = {}
        
        # Extract troubleshooting performed
        troubleshooting_patterns = [
            r'\*\*Troubleshooting Steps Performed:\*\*(.*?)(?=\*\*Mitigations:|---|\*\*|$)',
            r'- \*\*Troubleshooting Steps Performed:\*\*(.*?)(?=- \*\*|---|\*\*|$)'
        ]
        
        for pattern in troubleshooting_patterns:
            troubleshooting_match = re.search(pattern, content, re.DOTALL)
            if troubleshooting_match:
                steps_text = troubleshooting_match.group(1)
                steps = re.findall(r'-\s*(.+?)(?=\n\s*-|\n\*\*|$)', steps_text, re.DOTALL)
                data['troubleshooting_performed'] = [step.strip() for step in steps]
                break
        
        if 'troubleshooting_performed' not in data:
            data['troubleshooting_performed'] = []
        
        return data
    
    def _parse_executive_summary_section(self, content):
        """Parse Executive Summary section"""
        data = {}
        
        # Extract technical summary
        summary_patterns = [
            r'\*\*Summary:\*\*(.*?)(?=\*\*Recommendations:|---|\*\*|$)',
            r'### Technical Summary\n(.*?)(?=### |---|\*\*|$)'
        ]
        
        for pattern in summary_patterns:
            summary_match = re.search(pattern, content, re.DOTALL)
            if summary_match:
                data['technical_summary'] = summary_match.group(1).strip()
                break
        
        if 'technical_summary' not in data:
            data['technical_summary'] = ""
        
        # Extract recommendations
        recommendations_patterns = [
            r'\*\*Recommendations:\*\*(.*?)(?=\*\*Priority:|---|\*\*|$)',
            r'### Actionable Recommendations\n(.*?)(?=### |---|\*\*|$)'
        ]
        
        for pattern in recommendations_patterns:
            recommendations_match = re.search(pattern, content, re.DOTALL)
            if recommendations_match:
                recs_text = recommendations_match.group(1)
                recs = re.findall(r'-\s*(.+?)(?=\n\s*-|\n\*\*|$)', recs_text, re.DOTALL)
                data['actionable_recommendations'] = [rec.strip() for rec in recs]
                break
        
        if 'actionable_recommendations' not in data:
            data['actionable_recommendations'] = []
        
        return data
    
    def _parse_tool_invocations(self, content):
        """Parse tool invocation information from debug logs"""
        tools = {}
        
        # Extract tool invocations from debug logs
        tool_patterns = {
            'sighting_attachments': r'name: sighting_attachments\n\s+type: tool',
            'sighting_get_category': r'name: sighting_get_category\n\s+type: tool',
            'sighting_mandatory_dfd_analyzer': r'name: sighting_mandatory_dfd_analyzer\n\s+type: tool',
            'sighting_display_bkm': r'name: sighting_display_bkm\n\s+type: tool'
        }
        
        for tool_name, pattern in tool_patterns.items():
            tools[tool_name] = bool(re.search(pattern, content))
        
        return tools



    def _parse_processing_duration(self, content):
        
        """Parse processing duration from start and end timestamps"""
        duration_info = {}
        
        # Extract start timestamp from first line
        start_pattern = r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \[DEBUG\] GNAI Cli'
        start_match = re.search(start_pattern, content, re.MULTILINE)
        
        # Extract end timestamp from cleanup line
        end_pattern = r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \[DEBUG\] Cleaning up temporary workspace'
        end_match = re.search(end_pattern, content)
        
        if start_match and end_match:
            try:
                start_time_str = start_match.group(1)
                end_time_str = end_match.group(1)
                
                # Parse timestamps
                start_time = datetime.strptime(start_time_str, '%Y-%m-%d %H:%M:%S')
                end_time = datetime.strptime(end_time_str, '%Y-%m-%d %H:%M:%S')
                
                # Calculate duration
                duration = end_time - start_time
                duration_seconds = duration.total_seconds()
                
                duration_info = {
                    'start_time': start_time_str,
                    'end_time': end_time_str,
                    'duration_seconds': duration_seconds,
                    'duration_formatted': str(duration),
                    'duration_minutes': round(duration_seconds / 60, 2)
                }
                
            except ValueError as e:
                print(f"⚠️ Error parsing timestamps: {e}")
                duration_info = {
                    'start_time': start_time_str if start_match else 'Not found',
                    'end_time': end_time_str if end_match else 'Not found',
                    'error': str(e)
                }
        else:
            duration_info = {
                'start_time': 'Not found',
                'end_time': 'Not found',
                'error': 'Could not find start or end timestamps'
            }
        
        return duration_info


def create_usage_statistics_widget(usage_stats, processing_duration):
    # Helper function to safely format numbers with commas
    def format_number(value, default='N/A'):
        if value is None or value == 'N/A':
            return default
        try:
            return f"{int(value):,}"
        except (ValueError, TypeError):
            return default
    
    # Format the usage statistics HTML
    usage_html = f"""
    <div style="background-color: #f0f0f0; padding: 15px; border-radius: 5px; margin: 10px 0;">
        <h4>Processing Statistics</h4>
        <ul style="margin: 0; padding-left: 20px;">
            <li><strong>Processing Duration:</strong> {processing_duration:.2f} seconds</li>
            <li><strong>Prompt Tokens:</strong> {format_number(usage_stats.get('prompt_tokens'))}</li>
            <li><strong>Completion Tokens:</strong> {format_number(usage_stats.get('completion_tokens'))}</li>
            <li><strong>Total Tokens:</strong> {format_number(usage_stats.get('total_tokens'))}</li>
            <li><strong>Model:</strong> {usage_stats.get('model', 'N/A')}</li>
        </ul>
    </div>
    """
    
    return widgets.HTML(value=usage_html)



def create_usage_statistics_widget(usage_stats, processing_duration=None):
    """Create a widget to display usage statistics and processing time"""
    if not usage_stats and not processing_duration:
        return widgets.HTML(value="<p><em>No usage statistics or processing time available</em></p>")
    
    # Helper function to safely format numbers with commas
    def format_number(value, default='N/A'):
        if value is None or value == 'N/A':
            return default
        try:
            return f"{int(value):,}"
        except (ValueError, TypeError):
            return default
    
    # Helper function to safely format cost
    def format_cost(value, default='N/A'):
        if value is None or value == 'N/A':
            return default
        try:
            return f"{float(value):.4f}"
        except (ValueError, TypeError):
            return default
    
    # Format usage statistics as HTML
    cache_info = usage_stats.get('cache', {}) if usage_stats else {}
    
    # Build the processing time section
    processing_time_section = ""
    if processing_duration and 'duration_seconds' in processing_duration:
        processing_time_section = f"""
        <div style="margin-top: 10px;">
            <h5>⏱️ Processing Time</h5>
            <ul style="margin: 5px 0;">
                <li><strong>Start Time:</strong> {processing_duration.get('start_time', 'N/A')}</li>
                <li><strong>End Time:</strong> {processing_duration.get('end_time', 'N/A')}</li>
                <li><strong>Duration:</strong> {processing_duration.get('duration_formatted', 'N/A')}</li>
                <li><strong>Total Minutes:</strong> {processing_duration.get('duration_minutes', 'N/A')} min</li>
            </ul>
        </div>
        """
    
    stats_html = f"""
    <div style="background-color: #f8f9fa; 
                border: 2px solid #17a2b8; 
                border-radius: 8px; 
                padding: 15px; 
                margin: 10px 0;">
        <h4 style="color: #17a2b8; margin-top: 0;">💰 Usage Statistics</h4>
        
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px;">
            <div>
                <h5>📊 Token Usage</h5>
                <ul style="margin: 5px 0;">
                    <li><strong>Total Tokens:</strong> {format_number(usage_stats.get('total_tokens'))}</li>
                    <li><strong>Prompt Tokens:</strong> {format_number(usage_stats.get('prompt_tokens'))}</li>
                    <li><strong>Completion Tokens:</strong> {format_number(usage_stats.get('completion_tokens'))}</li>
                </ul>
            </div>
            
            <div>
                <h5>💸 Cost & Performance</h5>
                <ul style="margin: 5px 0;">
                    <li><strong>Total Cost:</strong> ${format_cost(usage_stats.get('total_cost'))}</li>
                    <li><strong>Successful Requests:</strong> {format_number(usage_stats.get('successful_requests'))}</li>
                </ul>
            </div>
        </div>
        
        {f'''
        <div style="margin-top: 10px;">
            <h5>🔄 Cache Performance</h5>
            <ul style="margin: 5px 0;">
                <li><strong>Cache Hits:</strong> {format_number(cache_info.get('cache_hits'))}</li>
                <li><strong>Hit Ratio:</strong> {cache_info.get('hit_ratio', 'N/A')}%</li>
                <li><strong>Cache Misses:</strong> {format_number(cache_info.get('misses'))}</li>
                <li><strong>Read Tokens:</strong> {format_number(cache_info.get('read_tokens'))}</li>
                <li><strong>Write Tokens:</strong> {format_number(cache_info.get('write_tokens'))}</li>
            </ul>
        </div>
        ''' if cache_info else ''}
        
        {processing_time_section}
    </div>
    """
    
    return widgets.HTML(value=stats_html)


# Rest of the functions with enhanced content display
def populate_dataframe_from_gnai_output(file_path, df):
    """Populate DataFrame from GNAI output file"""
    parser = GNAIOutputParser()
    parsed_data = parser.parse_output_file(file_path)
    
    if parsed_data is None:
        print("❌ Failed to parse the output file.")
        return df, None
    
    # Create a new row
    row_data = {}
    
    # Populate HSD ID - USE THE ALREADY EXTRACTED ID
    row_data[('HSD ID', '')] = parsed_data['hsd_id']  # This should use the already extracted ID
    
       
    # Populate Content Extraction & Summary
    structured = parsed_data.get('structured_output', {})
    content_extraction = structured.get('content_extraction', {})
    
    row_data[('Content Extraction & Summary', 'Title')] = content_extraction.get('title', '')
    row_data[('Content Extraction & Summary', 'Main issue')] = content_extraction.get('main_issue', '')
    row_data[('Content Extraction & Summary', 'Description')] = content_extraction.get('description', '')
    
    # ETL Attachment Info
    etl_attachments = content_extraction.get('etl_attachments', [])
    row_data[('Content Extraction & Summary', 'ETL Attachment Info: Names')] = '; '.join([att['name'] for att in etl_attachments])
    row_data[('Content Extraction & Summary', 'ETL Attachment Info: Types')] = '; '.join([att['type'] for att in etl_attachments])
    row_data[('Content Extraction & Summary', 'ETL Attachment Info: Notes')] = '; '.join([att['notes'] for att in etl_attachments])
    
    # Driver Info
    driver_info = parsed_data.get('driver_info', [])
    row_data[('Content Extraction & Summary', 'Driver Info: Build Types')] = '; '.join([d['build_type'] for d in driver_info])
    row_data[('Content Extraction & Summary', 'Driver Info: Versions')] = '; '.join([d['version'] for d in driver_info])
    row_data[('Content Extraction & Summary', 'Driver Info: Build Dates')] = '; '.join([d['build_date'] for d in driver_info])
    
    # Pipe Underrun Check
    pipe_info = parsed_data.get('pipe_underrun_info', {})
    if pipe_info.get('detected'):
        row_data[('Content Extraction & Summary', 'Pipe Underrun Check')] = f"Pipe underrun detected in {pipe_info['count']} ETL file(s)"
    else:
        row_data[('Content Extraction & Summary', 'Pipe Underrun Check')] = "No pipe underrun detected"
    
    # Reproduction Instructions
    repro_instructions = content_extraction.get('reproduction_instructions', [])
    row_data[('Content Extraction & Summary', 'Reproduction Instructions')] = '; '.join(repro_instructions)
    
    # System Configuration
    row_data[('Content Extraction & Summary', 'System Configuration')] = content_extraction.get('system_configuration', '')
    
    # Issue Classification
    issue_classification = structured.get('issue_classification', {})
    row_data[('Issue Classification', 'Category')] = issue_classification.get('category', '')
    row_data[('Issue Classification', 'Agent Output')] = issue_classification.get('reasoning', '')
    
    # Attachment Verification
    attachment_verification = structured.get('attachment_verification', {})
    row_data[('Attachment Verification', 'Validity Status')] = attachment_verification.get('validity_status', '')
    
    # DFD Checklist
    dfd_checklist = structured.get('dfd_checklist', {})
    checklist_items = dfd_checklist.get('mandatory_checklist', [])
    compliant_count = sum(1 for item in checklist_items if item.get('compliant', False))
    total_count = len(checklist_items)
    row_data[('Suggestions as per DFD checklist', 'Compliance Status')] = f"{compliant_count}/{total_count} compliant" if total_count > 0 else "No checklist found"
    
    # Tool Invocation Log
    tool_log = parsed_data.get('tool_log', {})
    row_data[('Tool Invocation Log', 'Sighting Attachments Invoked')] = str(tool_log.get('sighting_attachments', False))
    row_data[('Tool Invocation Log', 'Sighting Get Category Invoked')] = str(tool_log.get('sighting_get_category', False))
    row_data[('Tool Invocation Log', 'Mandatory DFD Analyzer Invoked')] = str(tool_log.get('sighting_mandatory_dfd_analyzer', False))
    row_data[('Tool Invocation Log', 'Category Specific BKM Tool Invoked')] = str(tool_log.get('sighting_display_bkm', False))
    
    # Executive Summary
    executive_summary = structured.get('executive_summary', {})
    row_data[('Executive Summary and Recommendations', 'Technical Summary')] = executive_summary.get('technical_summary', '')
    recommendations = executive_summary.get('actionable_recommendations', [])
    row_data[('Executive Summary and Recommendations', 'Actionable Recommendations')] = '; '.join(recommendations)
    
    # Fill remaining columns with empty strings
    for col in df.columns:
        if col not in row_data:
            row_data[col] = ''
    
    # Add row to DataFrame
    new_row = pd.DataFrame([row_data], columns=df.columns)
    df = pd.concat([df, new_row], ignore_index=True)
    
    return df, parsed_data

def create_accuracy_sliders_with_content(parsed_data):
    """Create sliders with full content display for each major section"""
    
    # Get section contents
    section_contents = parsed_data.get('section_contents', {})
    
    # Define sections with their content
    sections_config = [
        {
            'name': "Content Extraction & Summary",
            'content': section_contents.get('content_extraction', 'No content found for this section')
        },
        {
            'name': "Issue Classification",
            'content': section_contents.get('issue_classification', 'No content found for this section')
        },
        {
            'name': "Attachment Verification",
            'content': section_contents.get('attachment_verification', 'No content found for this section')
        },
        {
            'name': "DFD Checklist Compliance",
            'content': section_contents.get('dfd_checklist', 'No content found for this section')
        },
        {
            'name': "Triage and Troubleshooting Review",
            'content': section_contents.get('triage_review', 'No content found for this section')
        },
        {
            'name': "Executive Summary & Recommendations",
            'content': section_contents.get('executive_summary', 'No content found for this section')
        },
        {
            'name': "Overall Tool Performance",
            'content': f"""
## Overall Assessment

**HSD ID:** {parsed_data.get('hsd_id', 'Unknown')}

**Processing Summary:**
- Attachments found: {len(parsed_data.get('attachment_info', {}).get('attachment_list', []))}
- Driver configurations: {len(parsed_data.get('driver_info', []))}
- Pipe underrun detected: {parsed_data.get('pipe_underrun_info', {}).get('detected', False)}
- Structured sections parsed: {len(parsed_data.get('structured_output', {}))}
- Tools invoked: {sum(parsed_data.get('tool_log', {}).values())}

**Tool Invocations:**
{chr(10).join([f"- {tool}: {'✅' if invoked else '❌'}" for tool, invoked in parsed_data.get('tool_log', {}).items()])}

Rate the overall performance of the GNAI Sighting Assistant Tool considering:
- Completeness of information extraction
- Accuracy of analysis
- Quality of recommendations
- Usefulness for debugging workflow
- Overall time saved vs manual analysis
            """
        }
    ]
    
    sliders = {}
    
    # Display header
    display(HTML("""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                color: white; 
                padding: 20px; 
                border-radius: 10px; 
                margin: 20px 0;
                text-align: center;">
        <h1 style="margin: 0; font-size: 28px;">🎯 GNAI Sighting Assistant Tool Accuracy Assessment</h1>
        <p style="margin: 10px 0 0 0; font-size: 16px; opacity: 0.9;">
            Rate each section's accuracy and provide detailed feedback
        </p>
    </div>
    """))
    
    for section_config in sections_config:
        print(f"\n{'='*80}")
        print(f"Creating assessment for: {section_config['name']}")
        print('='*80)
        
        sliders[section_config['name']] = MakeSliderWithContent(
            name=section_config['name'],
            content=section_config['content'],
            initial=5,
            min_val=0,
            max_val=10
        )
    
    return sliders


def save_assessment_results(sliders, hsd_id, parsed_data=None, output_file=None):
    """Save slider results to JSON file with enhanced data including usage statistics"""
    

    # Save assessment results to data directory
    current_dir = os.getcwd()
    data_dir = os.path.join(current_dir, "data")
    os.makedirs(data_dir, exist_ok=True)

#######

    # Generate filename based on HSD ID if not provided
    if output_file is None:
        output_file = f"{hsd_id}_accuracy_assessment.json"
        output_file = os.path.join(data_dir, output_file)



    print(f"\n{'='*60}")
    print(f"💾 SAVING ASSESSMENT RESULTS")
    print(f"{'='*60}")
    print(f"{'='*60}")
    print(f"[TEST]THE OUTPUT FILE IS : {output_file}")


    results = {
        'hsd_id': hsd_id,
        'timestamp': datetime.now().isoformat(),
        'assessments': {},
        'summary': {
            'total_sections': len(sliders),
            'average_rating': 0,
            'sections_with_feedback': 0
        }
    }
    
    # Add usage statistics if available
    if parsed_data and 'usage_statistics' in parsed_data:
        results['usage_statistics'] = parsed_data['usage_statistics']
        print(f"📊 Usage statistics included in assessment")
    
    # Add processing duration if available
    if parsed_data and 'processing_duration' in parsed_data:
        results['processing_duration'] = parsed_data['processing_duration']
        duration_info = parsed_data['processing_duration']
        if 'duration_seconds' in duration_info:
            print(f"⏱️ Processing duration included: {duration_info['duration_formatted']}")
    
    total_rating = 0
    sections_with_feedback = 0
    
    print(f"📝 Processing {len(sliders)} section assessments...")
    
    for section_name, slider in sliders.items():
        slider_data = slider.get_data()
        results['assessments'][section_name] = slider_data
        total_rating += slider_data['value']
        if slider_data['remark'].strip():
            sections_with_feedback += 1
        print(f"   ✓ {section_name}: {slider_data['value']}/10 {'(with feedback)' if slider_data['remark'].strip() else ''}")
    
    results['summary']['average_rating'] = round(total_rating / len(sliders), 2)
    results['summary']['sections_with_feedback'] = sections_with_feedback
    
    # Check if file exists (for informational purposes)
    file_exists = os.path.exists(output_file)
    if file_exists:
        print(f"📝 File {output_file} exists - will be overwritten")
    else:
        print(f"📄 Creating new assessment file: {output_file}")
    
    # Save the file (overwrite mode)
    try:
        with open(output_file, 'w') as f:
            json.dump(results, f, indent=2)
        
        print(f"\n🎉 SUCCESS! Assessment results saved successfully!")
        print(f"📁 Output file: {os.path.abspath(output_file)}")
        print(f"📊 Assessment Summary:")
        print(f"   • HSD ID: {hsd_id}")
        print(f"   • Average rating: {results['summary']['average_rating']}/10")
        print(f"   • Sections with feedback: {sections_with_feedback}/{len(sliders)}")
        print(f"   • File mode: {'Overwritten' if file_exists else 'Created new'}")
        
        # Print usage statistics if available
        if 'usage_statistics' in results:
            stats = results['usage_statistics']
            print(f"💰 Usage Statistics:")
            print(f"   • Total cost: ${stats.get('total_cost', 'N/A'):.4f}")
            print(f"   • Total tokens: {stats.get('total_tokens', 'N/A'):,}")
            print(f"   • Successful requests: {stats.get('successful_requests', 'N/A')}")
            if 'cache' in stats:
                cache = stats['cache']
                print(f"   • Cache hit ratio: {cache.get('hit_ratio', 'N/A')}%")
        
        # Print processing duration if available
        if 'processing_duration' in results:
            duration_info = results['processing_duration']
            if 'duration_seconds' in duration_info:
                print(f"⏱️ Processing Duration: {duration_info['duration_formatted']} ({duration_info['duration_minutes']} minutes)")
        
        print(f"{'='*60}")
        
    except Exception as e:
        print(f"❌ ERROR: Failed to save assessment results!")
        print(f"   Error details: {e}")
        print(f"   File path: {os.path.abspath(output_file)}")
        raise e


def process_gnai_output(file_path):
    """Complete workflow to process GNAI output and create assessment interface"""
    
    print(f"🚀 Processing GNAI output file: {file_path}")
    print("="*80)
    
    # Parse and populate DataFrame
    global df
    df, parsed_data = populate_dataframe_from_gnai_output(file_path, df)
    
    if parsed_data is None:
        print("❌ Failed to process the file.")
        return None, None, None
    

    # FORCE CORRECT HSD ID IF IT'S WRONG
    if parsed_data.get('hsd_id') == 'Unknown':
        print("🔧 Fixing HSD ID extraction...")
        # Re-read the file and extract ID
        content = read_file_with_fallback_encoding(file_path)
        if content:
            parser = GNAIOutputParser()
            correct_id = parser._extract_hsd_id(content)
            parsed_data['hsd_id'] = correct_id
            print(f"✅ Fixed HSD ID: {correct_id}")


    print("✅ DataFrame populated successfully!")
    print(f"DataFrame shape: {df.shape}")
    
    # Get HSD ID for filename
    hsd_id = parsed_data.get('hsd_id', 'Unknown')

    # Save assessment results to data directory
    current_dir = os.getcwd()
    data_dir = os.path.join(current_dir, "data")
    os.makedirs(data_dir, exist_ok=True)
    
    # Update the assessment results file path to data directory with correct format
    output_filename = os.path.join(data_dir, f"{hsd_id}_assessment_results.json")

    # Display usage statistics if available
    usage_stats = parsed_data.get('usage_statistics', {})
    processing_duration = parsed_data.get('processing_duration', {})
    
    if usage_stats:
        print("\n💰 USAGE STATISTICS:")
        print("-" * 40)
        print(f"Total Cost: ${usage_stats.get('total_cost', 'N/A'):.4f}")
        print(f"Total Tokens: {usage_stats.get('total_tokens', 'N/A'):,}")
        print(f"Successful Requests: {usage_stats.get('successful_requests', 'N/A')}")
        if 'cache' in usage_stats:
            cache = usage_stats['cache']
            print(f"Cache Hit Ratio: {cache.get('hit_ratio', 'N/A')}%")
    
    # Display processing duration if available
    if processing_duration and 'duration_seconds' in processing_duration:
        print(f"\n⏱️ PROCESSING TIME:")
        print("-" * 40)
        print(f"Start Time: {processing_duration.get('start_time', 'N/A')}")
        print(f"End Time: {processing_duration.get('end_time', 'N/A')}")
        print(f"Duration: {processing_duration.get('duration_formatted', 'N/A')} ({processing_duration.get('duration_minutes', 'N/A')} minutes)")
    
    # Display summary of parsed data
    print("\n📊 PARSING SUMMARY:")
    print("-" * 40)
    print(f"HSD ID: {parsed_data['hsd_id']}")
    print(f"Output filename: {output_filename}")
    print(f"Attachments found: {len(parsed_data['attachment_info']['attachment_list'])}")
    print(f"Driver configurations: {len(parsed_data['driver_info'])}")
    print(f"Pipe underrun detected: {parsed_data['pipe_underrun_info']['detected']}")
    print(f"Structured sections parsed: {len(parsed_data['structured_output'])}")
    print(f"Tools invoked: {sum(parsed_data['tool_log'].values())}")
    
    # Create accuracy assessment sliders with content
    print("\n" + "="*80)
    print("🎯 CREATING ACCURACY ASSESSMENT WITH CONTENT DISPLAY")
    print("="*80)
    
    sliders = create_accuracy_sliders_with_content(parsed_data)
    
    # Display usage statistics widget with processing time
    if usage_stats or processing_duration:
        usage_widget = create_usage_statistics_widget(usage_stats, processing_duration)
        display(usage_widget)
    
    # Create enhanced save button with output widget for feedback
    save_output = widgets.Output()
    
    save_button = widgets.Button(
        description=f"💾 Save Assessment ({hsd_id})", 
        button_style='success',
        layout=widgets.Layout(width='350px', height='50px'),
        style={'font_weight': 'bold'}
    )
    
    def on_save_clicked(b):
        with save_output:
            save_output.clear_output()
            print(f"🔄 Saving assessment results to {output_filename}...")
            
        try:
            save_assessment_results(sliders, hsd_id, parsed_data, output_filename)
            
            # Update button appearance
            save_button.description = f"✅ Saved to {output_filename}!"
            save_button.button_style = 'info'
            save_button.disabled = True
            
            with save_output:
                print(f"\n🎊 Assessment successfully saved to {output_filename}!")
                print("📂 You can find the file in your current working directory.")
                print(f"📍 Full path: {os.path.abspath(output_filename)}")
                
        except Exception as e:
            save_button.description = "❌ Save Failed"
            save_button.button_style = 'danger'
            with save_output:
                print(f"❌ Error saving assessment: {e}")
    
    save_button.on_click(on_save_clicked)
    
    # Create summary widget with filename info
    summary_html = f"""
    <div style="background-color: #f8f9fa; 
                border: 2px solid #28a745; 
                border-radius: 8px; 
                padding: 20px; 
                margin: 20px 0;
                text-align: center;">
        <h3 style="color: #28a745; margin-top: 0;">📋 Assessment Instructions</h3>
        <p><strong>1.</strong> Review each section's content by expanding the accordion</p>
        <p><strong>2.</strong> Rate accuracy from 0 (completely wrong) to 10 (perfect)</p>
        <p><strong>3.</strong> Provide detailed feedback in the text areas</p>
        <p><strong>4.</strong> Click the save button when complete</p>
        <hr style="margin: 15px 0;">
        <p style="margin-bottom: 0;">
            <strong>HSD ID:</strong> {parsed_data['hsd_id']} | 
            <strong>Sections:</strong> {len(parsed_data.get('section_contents', {}))} | 
            <strong>Tools Used:</strong> {sum(parsed_data['tool_log'].values())}
        </p>
        <p style="margin: 5px 0 0 0; font-size: 14px; color: #666;">
            📁 Will save as: <code>{output_filename}</code>
        </p>
    </div>
    """
    
    display(HTML(summary_html))
    display(save_button)
    display(save_output)  # Display the output widget for save feedback
    
    return df, parsed_data, sliders


#### Run SAT gnai CLI command Related Code

In [11]:

def run_gnai_command(hsd_id):
    """Run the GNAI PowerShell command with compatible encoding handling"""
    
    current_dir = os.getcwd()
    
    # Create runs directory if it doesn't exist
    runs_dir = os.path.join(current_dir, "runs")
    os.makedirs(runs_dir, exist_ok=True)
    
    # Set output file path in runs directory with HSD ID
    output_file = f"{hsd_id}_run.txt"
    output_path = os.path.join(runs_dir, output_file)
    
    powershell_command = f"""
    $OutputEncoding = [System.Text.Encoding]::UTF8
    $output = dt gnai ask 'Give summary of HSD id {hsd_id}' -v --assistant sighting_assistant
    $output | Out-File -FilePath '{output_path}' -Encoding UTF8
    Write-Host $output
    """
    
    print(f"🚀 RUNNING GNAI SIGHTING ASSISTANT")
    print("=" * 80)
    print(f"📋 HSD ID: {hsd_id}")
    print(f"📁 Output file: {output_path}")
    print(f"⚡ Command: dt gnai ask 'Give summary of HSD id {hsd_id}' -v --assistant sighting_assistant")
    print("=" * 80)
    
    try:
        start_time = datetime.now()
        print(f"⏰ Started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print("🔄 Running command... (this may take several minutes)")
        
        # Run with proper encoding handling
        result = subprocess.run(
            ["powershell", "-Command", powershell_command],
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='replace',
            timeout=600
        )
        
        end_time = datetime.now()
        duration = end_time - start_time
        
        print(f"⏰ Completed at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"⏱️ Duration: {duration}")
        
        if result.returncode == 0:
            print("✅ Command executed successfully!")
            
            if os.path.exists(output_path):
                file_size = os.path.getsize(output_path)
                print(f"📄 Output file created: {output_path}")
                print(f"📊 File size: {file_size:,} bytes")
                
                # Show command output in console
                if result.stdout:
                    print("📺 Command output preview:")
                    print(result.stdout[:500] + "..." if len(result.stdout) > 500 else result.stdout)
                
                return True, output_path, None
            else:
                error_msg = "Command succeeded but output file was not created"
                print(f"❌ {error_msg}")
                return False, output_path, error_msg
        else:
            error_msg = f"Command failed with return code {result.returncode}"
            print(f"❌ {error_msg}")
            if result.stderr:
                print(f"📝 STDERR: {result.stderr}")
            if result.stdout:
                print(f"📝 STDOUT: {result.stdout}")
            return False, output_path, error_msg
            
    except subprocess.TimeoutExpired:
        error_msg = "Command timed out after 10 minutes"
        print(f"⏰ {error_msg}")
        return False, output_path, error_msg
        
    except Exception as e:
        error_msg = f"Unexpected error: {str(e)}"
        print(f"❌ {error_msg}")
        return False, output_path, error_msg

def run_complete_workflow(hsd_id):
    """Run GNAI command and then process the results"""
    
    print("🚀 COMPLETE GNAI WORKFLOW")
    print("=" * 80)
    
    # Step 1: Run GNAI command
    success, output_path, error = run_gnai_command(hsd_id)
    
    if not success:
        print(f"❌ Workflow stopped: {error}")
        return None, None, None
    
    # Step 2: Wait a moment for file to be fully written
    print("⏳ Waiting for file to be ready...")
    time.sleep(2)
    
    # Step 3: Process the output
    print("\n" + "=" * 80)
    print("🎯 STARTING ACCURACY ASSESSMENT")
    print("=" * 80)
    
    try:
        df, parsed_data, sliders = process_gnai_output(output_path)
        
        # Save assessment results to data directory
        current_dir = os.getcwd()
        data_dir = os.path.join(current_dir, "data")
        os.makedirs(data_dir, exist_ok=True)
        
        # Update the assessment results file path to data directory with correct format
        assessment_file = os.path.join(data_dir, f"{hsd_id}_assessment_results.json")
        
        # If df exists, save it to the data directory as JSON
        #if df is not None:
            #df.to_json(assessment_file, orient='records', indent=2)
            #print(f"📊 Assessment results will be saved to: {assessment_file}")
        
        return df, parsed_data, sliders
    except Exception as e:
        print(f"❌ Error in processing: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None



---

#### Main Orchestrator Code and Input Feedback here

In [12]:
import sys
import os
import subprocess
import time
from datetime import datetime
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import HTML, display



# Use current working directory as the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

from src.hsdes import HSDESAPI

# Optimize display for long outputs
display(HTML("""
<style>
    .output_area {
        max-height: 85vh !important;
        overflow-y: auto !important;
        font-size: 11px !important;
    }
    .container { width: 98% !important; }
    .jp-OutputArea { max-height: 85vh !important; overflow-y: auto !important; }
    .output_subarea { max-height: none !important; }
    
    /* Progress indicators */
    .progress-bar {
        width: 100%;
        background-color: #f0f0f0;
        border-radius: 5px;
        margin: 5px 0;
    }
    .progress-fill {
        height: 20px;
        background-color: #4CAF50;
        border-radius: 5px;
        text-align: center;
        line-height: 20px;
        color: white;
        font-weight: bold;
    }
</style>
"""))

# HSDES API Integration
# Use current working directory as the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

print("📁 Current working directory:", os.getcwd())
print("📁 Project root:", project_root)

try:
    from src.hsdes import HSDESAPI
    print("✅ HSDES API imported successfully")
except ImportError as e:
    print(f"❌ Failed to import HSDES API: {e}")
    print("Please ensure the src/hsdes module is available")



# Threading configuration
MAX_CONCURRENT_THREADS = 3

def validate_user_input():
    """Validate and process user input configuration"""
    
    print("🎯 USER INPUT VALIDATION")
    print("=" * 80)
    
    if USER_INPUT_TYPE.lower() == "query":
        if not QUERY_ID or QUERY_ID == "your_query_id_here":
            print("❌ Error: QUERY_ID is not set properly")
            print("Please set QUERY_ID to your actual query ID")
            return False, None, None
        
        print(f"✅ Input Type: Query ID")
        print(f"🔍 Query ID: {QUERY_ID}")
        return True, "query", QUERY_ID
        
    elif USER_INPUT_TYPE.lower() == "hsd_id":
        try:
            single_hsd_id = globals().get('SINGLE_HSD_ID')
            if not single_hsd_id:
                print("❌ Error: SINGLE_HSD_ID is not set")
                print("Please set SINGLE_HSD_ID to your HSD ID")
                return False, None, None
            
            print(f"✅ Input Type: Single HSD ID")
            print(f"📋 HSD ID: {single_hsd_id}")
            return True, "hsd_id", str(single_hsd_id)
            
        except NameError:
            print("❌ Error: SINGLE_HSD_ID is not defined")
            print("Please uncomment and set SINGLE_HSD_ID in the configuration")
            return False, None, None
    
    else:
        print(f"❌ Error: Invalid USER_INPUT_TYPE '{USER_INPUT_TYPE}'")
        print("Please set USER_INPUT_TYPE to either 'query' or 'hsd_id'")
        return False, None, None

def get_hsd_ids_from_query(query_id, limit=5):
    """Retrieve HSD IDs from HSDES query with optional limit"""
    print(f"🔍 Retrieving HSD IDs from query: {query_id}")
    
    try:
        hsd = HSDESAPI()
        status, data = hsd.retrieve_article_ids_from_query(query_id)
        
        if status and data:
            total_count = len(data)
            print(f"✅ Successfully retrieved {total_count} HSD IDs from query")
            
            # Apply limit and inform user
            if total_count > limit:
                limited_data = data[:limit]
                print(f"⚠️  LIMITING RESULTS: Processing only the first {limit} HSD IDs out of {total_count} total")
                print(f"📋 This is done to manage processing time and system resources")
                print(f"📋 Selected HSD IDs: {', '.join(map(str, limited_data))}")
                print(f"📋 Skipped HSD IDs: {total_count - limit} items")
                return True, [str(hsd_id) for hsd_id in limited_data]
            else:
                print(f"📋 Processing all {total_count} HSD IDs from query")
                print(f"📋 HSD IDs: {', '.join(map(str, data))}")
                return True, [str(hsd_id) for hsd_id in data]
        else:
            print(f"❌ Failed to retrieve HSD IDs from query")
            return False, []
            
    except Exception as e:
        print(f"❌ Error retrieving HSD IDs: {e}")
        return False, []

def determine_hsd_ids():
    """Determine which HSD IDs to process based on user input"""
    
    # Validate user input first
    is_valid, input_type, input_value = validate_user_input()
    
    if not is_valid:
        return []
    
    if input_type == "query":
        print("🎯 Processing HSDES Query to retrieve HSD IDs")
        print("=" * 80)
        success, hsd_ids = get_hsd_ids_from_query(input_value, limit=6)  # Limit to 10

        # [TEST]
        
        if success and hsd_ids:
            print("=" * 80)
            print(f"📊 QUERY PROCESSING SUMMARY:")
            print(f"   🔍 Query ID: {input_value}")
            print(f"   📋 HSD IDs to process: {len(hsd_ids)}")
            print(f"   📝 Selected IDs: {', '.join(hsd_ids)}")
            print("=" * 80)
            return hsd_ids
        else:
            print("❌ Failed to retrieve HSD IDs from query")
            return []
            
    elif input_type == "hsd_id":
        print("🎯 Processing single HSD ID")
        print("=" * 80)
        print(f"📋 Single HSD ID: {input_value}")
        print("=" * 80)
        return [input_value]
    
    return []

def show_progress(current, total, task="Processing"):
    """Display a progress bar"""
    percentage = (current / total) * 100
    display(HTML(f"""
    <div class="progress-bar">
        <div class="progress-fill" style="width: {percentage}%">
            {task}: {current}/{total} ({percentage:.1f}%)
        </div>
    </div>
    """))

def run_gnai_command(hsd_id):
    """Run the GNAI PowerShell command with compatible encoding handling"""
    
    current_dir = os.getcwd()
    
    # Create runs directory if it doesn't exist
    runs_dir = os.path.join(current_dir, "runs")
    os.makedirs(runs_dir, exist_ok=True)
    
    # Set output file path in runs directory with HSD ID
    output_file = f"{hsd_id}_run.txt"
    output_path = os.path.join(runs_dir, output_file)
    
    # Use Out-File instead of Tee-Object for better encoding control
    powershell_command = f"""
    $OutputEncoding = [System.Text.Encoding]::UTF8
    $output = dt gnai ask 'Give summary of HSD id {hsd_id}' -v --assistant sighting_assistant
    $output | Out-File -FilePath '{output_path}' -Encoding UTF8
    Write-Host $output
    """
    
    thread_id = threading.current_thread().name
    print(f"🚀 [Thread {thread_id}] RUNNING GNAI SIGHTING ASSISTANT")
    print("=" * 80)
    print(f"📋 [Thread {thread_id}] HSD ID: {hsd_id}")
    print(f"📁 [Thread {thread_id}] Output file: {output_path}")
    print(f"⚡ [Thread {thread_id}] Command: dt gnai ask 'Give summary of HSD id {hsd_id}' -v --assistant sighting_assistant")
    print("=" * 80)
    
    try:
        start_time = datetime.now()
        print(f"⏰ [Thread {thread_id}] Started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"🔄 [Thread {thread_id}] Running command... (this may take several minutes)")
        
        # Run with proper encoding handling
        result = subprocess.run(
            ["powershell", "-Command", powershell_command],
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='replace',
            timeout=600
        )
        
        end_time = datetime.now()
        duration = end_time - start_time
        
        print(f"⏰ [Thread {thread_id}] Completed at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"⏱️ [Thread {thread_id}] Duration: {duration}")
        
        if result.returncode == 0:
            print(f"✅ [Thread {thread_id}] Command executed successfully!")
            
            if os.path.exists(output_path):
                file_size = os.path.getsize(output_path)
                print(f"📄 [Thread {thread_id}] Output file created: {output_path}")
                print(f"📊 [Thread {thread_id}] File size: {file_size:,} bytes")
                
                # Show command output in console (truncated for threading)
                if result.stdout:
                    print(f"📺 [Thread {thread_id}] Command output preview:")
                    print(result.stdout[:200] + "..." if len(result.stdout) > 200 else result.stdout)
                
                return True, output_path, None, hsd_id
            else:
                error_msg = "Command succeeded but output file was not created"
                print(f"❌ [Thread {thread_id}] {error_msg}")
                return False, output_path, error_msg, hsd_id
        else:
            error_msg = f"Command failed with return code {result.returncode}"
            print(f"❌ [Thread {thread_id}] {error_msg}")
            if result.stderr:
                print(f"📝 [Thread {thread_id}] STDERR: {result.stderr[:200]}...")
            return False, output_path, error_msg, hsd_id
            
    except subprocess.TimeoutExpired:
        error_msg = "Command timed out after 10 minutes"
        print(f"⏰ [Thread {thread_id}] {error_msg}")
        return False, output_path, error_msg, hsd_id
        
    except Exception as e:
        error_msg = f"Unexpected error: {str(e)}"
        print(f"❌ [Thread {thread_id}] {error_msg}")
        return False, output_path, error_msg, hsd_id


def process_single_hsd_assessment(hsd_id, output_path):
    """Process the assessment for a single HSD ID (runs in main thread)"""
    
    print(f"\n🎯 STARTING ACCURACY ASSESSMENT FOR HSD {hsd_id}")
    print("=" * 80)
    
    try:
        df, parsed_data, sliders = process_gnai_output(output_path)
        
        # The structured assessment results will be saved automatically by process_gnai_output()
        # when the user completes the interactive assessment
        
        # The file will be saved as: {hsd_id}_assessment_results.json in the data directory
        current_dir = os.getcwd()
        data_dir = os.path.join(current_dir, "data")
        os.makedirs(data_dir, exist_ok=True)
        
        # Expected assessment file path (created by the interactive assessment)
        assessment_file = os.path.join(data_dir, f"{hsd_id}_assessment_results.json")
        
        
        print(f"📊 Assessment results will be saved to: {assessment_file}")
        print(f"📝 Complete the interactive assessment to generate the structured results file")
        
        return df, parsed_data, sliders, hsd_id, assessment_file
    except Exception as e:
        print(f"❌ Error in processing HSD {hsd_id}: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None, hsd_id, None




def run_concurrent_gnai_commands(hsd_ids, max_workers=None):
    """Run GNAI commands concurrently for multiple HSD IDs"""
    
    if max_workers is None:
        max_workers = min(MAX_CONCURRENT_THREADS, len(hsd_ids))
    
    print("🚀 CONCURRENT GNAI WORKFLOW")
    print("=" * 100)
    print(f"📋 Processing {len(hsd_ids)} HSD IDs concurrently with {max_workers} threads")
    print(f"📋 HSD IDs: {', '.join(hsd_ids)}")
    print("=" * 100)
    
    completed_commands = {}
    failed_commands = {}
    
    # Use ThreadPoolExecutor for concurrent execution
    with ThreadPoolExecutor(max_workers=max_workers, thread_name_prefix="GNAI") as executor:
        # Submit all GNAI commands
        future_to_hsd = {executor.submit(run_gnai_command, hsd_id): hsd_id for hsd_id in hsd_ids}
        
        # Process completed commands as they finish
        completed_count = 0
        for future in as_completed(future_to_hsd):
            hsd_id = future_to_hsd[future]
            completed_count += 1
            
            # Show progress
            show_progress(completed_count, len(hsd_ids), "GNAI Commands")
            
            try:
                success, output_path, error, returned_hsd_id = future.result()
                
                if success:
                    completed_commands[hsd_id] = {
                        'output_path': output_path,
                        'status': 'completed',
                        'run_file': f"\\runs\\{hsd_id}_run.txt"
                    }
                    print(f"✅ GNAI command completed for HSD {hsd_id}")
                else:
                    failed_commands[hsd_id] = {
                        'error': error,
                        'status': 'failed',
                        'run_file': f"\\runs\\{hsd_id}_run.txt"
                    }
                    print(f"❌ GNAI command failed for HSD {hsd_id}: {error}")
                    
            except Exception as exc:
                failed_commands[hsd_id] = {
                    'error': str(exc),
                    'status': 'error',
                    'run_file': None
                }
                print(f"💥 GNAI command error for HSD {hsd_id}: {exc}")
    
    print(f"\n🎯 GNAI COMMANDS SUMMARY:")
    print(f"   ✅ Completed: {len(completed_commands)}")
    print(f"   ❌ Failed: {len(failed_commands)}")
    
    return completed_commands, failed_commands

def run_sequential_assessments(completed_commands):
    """Run assessments sequentially for all completed GNAI commands"""
    
    print(f"\n🎯 STARTING SEQUENTIAL ASSESSMENTS")
    print("=" * 100)
    print(f"📊 Processing {len(completed_commands)} assessments...")
    
    assessment_results = {}
    
    for index, (hsd_id, command_result) in enumerate(completed_commands.items()):
        print(f"\n📋 ASSESSMENT {index + 1} of {len(completed_commands)}: HSD {hsd_id}")
        print("=" * 80)
        
        # Show progress
        show_progress(index + 1, len(completed_commands), "Assessments")
        
        # Wait a moment for file to be fully written
        print("⏳ Waiting for file to be ready...")
        time.sleep(2)
        
        # Process the assessment
        df, parsed_data, sliders, processed_hsd_id, assessment_file = process_single_hsd_assessment(
            hsd_id, command_result['output_path']
        )
        
        if df is not None:
            assessment_results[hsd_id] = {
                'status': 'success',
                'df': df,
                'parsed_data': parsed_data,
                'sliders': sliders,
                'assessment_file': assessment_file,
                'run_file': command_result['run_file']
            }
            print(f"✅ Assessment completed for HSD {hsd_id}")
        else:
            assessment_results[hsd_id] = {
                'status': 'failed',
                'df': None,
                'parsed_data': None,
                'sliders': None,
                'assessment_file': None,
                'run_file': command_result['run_file']
            }
            print(f"❌ Assessment failed for HSD {hsd_id}")
        
        # Add separator between assessments
        if index < len(completed_commands) - 1:
            print("\n" + "📊" * 30 + " NEXT ASSESSMENT " + "📊" * 30)
    
    return assessment_results

def run_flexible_workflow():
    """Main workflow function that handles both query ID and single HSD ID inputs"""
    
    print("🚀 FLEXIBLE HSDES + GNAI WORKFLOW")
    print("=" * 100)
    
    # Step 1: Determine HSD IDs based on user input
    hsd_ids = determine_hsd_ids()
    
    if not hsd_ids:
        print("❌ No HSD IDs available for processing")
        print("Please check your configuration and try again")
        return {}
    
    # Show processing plan
    print(f"\n🎯 PROCESSING PLAN:")
    print("=" * 80)
    print(f"📋 Total HSD IDs to process: {len(hsd_ids)}")
    print(f"📝 HSD IDs: {', '.join(hsd_ids)}")
    
    if USER_INPUT_TYPE.lower() == "query":
        print(f"🔍 Source: HSDES Query (limited to first 5 results)")
        print(f"⚡ Processing mode: {'Single-threaded' if len(hsd_ids) == 1 else 'Multi-threaded'}")
    else:
        print(f"📝 Source: Single HSD ID")
        print(f"⚡ Processing mode: Single-threaded")
    
    print(f"🧵 Max concurrent threads: {MAX_CONCURRENT_THREADS}")
    print("=" * 80)
    
    # Ask for user confirmation if processing multiple IDs
    if len(hsd_ids) > 1:
        print(f"\n⚠️  You are about to process {len(hsd_ids)} HSD IDs")
        print(f"⏱️  Estimated time: {len(hsd_ids) * 3} - {len(hsd_ids) * 5} minutes")
        print(f"📁 This will create {len(hsd_ids)} GNAI output files and {len(hsd_ids)} assessment files")
        print("=" * 80)
    
    start_time = datetime.now()
    
    # Step 2: Run GNAI commands (concurrent for multiple, single thread for one)
    if len(hsd_ids) == 1:
        print("🎯 Single HSD ID detected - running in single-threaded mode")
        hsd_id = hsd_ids[0]
        success, output_path, error, returned_hsd_id = run_gnai_command(hsd_id)
        
        if success:
            completed_commands = {hsd_id: {'output_path': output_path, 'status': 'completed', 'run_file': f"\\runs\\{hsd_id}_run.txt"}}
            failed_commands = {}
        else:
            completed_commands = {}
            failed_commands = {hsd_id: {'error': error, 'status': 'failed', 'run_file': f"\\runs\\{hsd_id}_run.txt"}}
    else:
        print(f"🎯 Multiple HSD IDs detected - running in multi-threaded mode")
        completed_commands, failed_commands = run_concurrent_gnai_commands(hsd_ids, MAX_CONCURRENT_THREADS)
    
    # Step 3: Run assessments sequentially
    assessment_results = {}
    if completed_commands:
        assessment_results = run_sequential_assessments(completed_commands)
    
    # Step 4: Combine results
    all_results = {}
    
    # Add successful assessments
    for hsd_id, result in assessment_results.items():
        all_results[hsd_id] = result
    
    # Add failed GNAI commands
    for hsd_id, result in failed_commands.items():
        all_results[hsd_id] = {
            'status': 'gnai_failed',
            'error': result['error'],
            'df': None,
            'parsed_data': None,
            'sliders': None,
            'assessment_file': None,
            'run_file': result['run_file']
        }
    
    # Final summary
    end_time = datetime.now()
    total_duration = end_time - start_time
    
    successful_assessments = len([r for r in all_results.values() if r['status'] == 'success'])
    failed_assessments = len([r for r in all_results.values() if r['status'] != 'success'])
    
    print("\n" + "🎊" * 100)
    print("🎊 FLEXIBLE WORKFLOW COMPLETE!")
    print("🎊" * 100)
    print(f"⏱️ Total Duration: {total_duration}")
    print(f"📊 SUMMARY:")
    print(f"   📋 Total HSD IDs processed: {len(hsd_ids)}")
    print(f"   ✅ Successful assessments: {successful_assessments}")
    print(f"   ❌ Failed assessments: {failed_assessments}")
    
    # Show input source with limitation info
    if USER_INPUT_TYPE.lower() == "query":
        print(f"   🔍 Source: HSDES Query ID {QUERY_ID} (limited to first 5 results)")
    else:
        print(f"   📝 Source: Single HSD ID")
    
    print(f"\n📁 FILES CREATED:")
    for hsd_id, result in all_results.items():
        if result['status'] == 'success':
            print(f"   ✅ {hsd_id}:")
            print(f"      📄 GNAI output: {result['run_file']}")
            print(f"      📊 Assessment: {result['assessment_file']}")
        else:
            status_msg = "GNAI command failed" if result['status'] == 'gnai_failed' else "Assessment failed"
            print(f"   ❌ {hsd_id}: {status_msg}")
            if result.get('run_file'):
                print(f"      📄 GNAI output: {result['run_file']}")
    
    print(f"\n⭐ Please complete the accuracy assessment for each successful HSD ID")
    
    return all_results

# Display current configuration with limitation info
print("🎯 CURRENT CONFIGURATION:")
print("=" * 80)
print(f"📋 Input Type: {USER_INPUT_TYPE}")
if USER_INPUT_TYPE.lower() == "query":
    print(f"🔍 Query ID: {QUERY_ID}")
    print(f"⚠️  Query Limit: First 5 HSD IDs only (to manage processing time)")
elif USER_INPUT_TYPE.lower() == "hsd_id":
    try:
        print(f"📝 Single HSD ID: {SINGLE_HSD_ID}")
    except NameError:
        print("❌ SINGLE_HSD_ID not defined - please set it in configuration")
print(f"🧵 Max Threads: {MAX_CONCURRENT_THREADS}")
print("=" * 80)

# Run the flexible workflow
results = run_flexible_workflow()

# Display final results summary
print(f"\n📋 FINAL RESULTS SUMMARY:")
for hsd_id, result in results.items():
    if result['status'] == 'success':
        print(f"✅ {hsd_id}: SUCCESS")
    elif result['status'] == 'gnai_failed':
        print(f"❌ {hsd_id}: GNAI COMMAND FAILED")
    else:
        print(f"⚠️ {hsd_id}: ASSESSMENT FAILED")


📁 Current working directory: C:\Users\tsontena\Documents\sighting\accuracy_test
📁 Project root: C:\Users\tsontena\Documents\sighting
✅ HSDES API imported successfully
🎯 CURRENT CONFIGURATION:
📋 Input Type: hsd_id
📝 Single HSD ID: 14026454583
🧵 Max Threads: 3
🚀 FLEXIBLE HSDES + GNAI WORKFLOW
🎯 USER INPUT VALIDATION
✅ Input Type: Single HSD ID
📋 HSD ID: 14026454583
🎯 Processing single HSD ID
📋 Single HSD ID: 14026454583

🎯 PROCESSING PLAN:
📋 Total HSD IDs to process: 1
📝 HSD IDs: 14026454583
📝 Source: Single HSD ID
⚡ Processing mode: Single-threaded
🧵 Max concurrent threads: 3
🎯 Single HSD ID detected - running in single-threaded mode
🚀 [Thread MainThread] RUNNING GNAI SIGHTING ASSISTANT
📋 [Thread MainThread] HSD ID: 14026454583
📁 [Thread MainThread] Output file: C:\Users\tsontena\Documents\sighting\accuracy_test\runs\14026454583_run.txt
⚡ [Thread MainThread] Command: dt gnai ask 'Give summary of HSD id 14026454583' -v --assistant sighting_assistant
⏰ [Thread MainThread] Started at: 2025

⏳ Waiting for file to be ready...

🎯 STARTING ACCURACY ASSESSMENT FOR HSD 14026454583
🚀 Processing GNAI output file: C:\Users\tsontena\Documents\sighting\accuracy_test\runs\14026454583_run.txt
Attempting to read file: C:\Users\tsontena\Documents\sighting\accuracy_test\runs\14026454583_run.txt
Trying encoding: UTF-8-SIG
✅ Successfully read file with encoding: UTF-8-SIG
File size: 28650 characters
✅ File loaded successfully. Content length: 28650 characters
🔍 Searching for HSD ID in content...
   Trying pattern 1: \*\*ID:\*\*\s*(\d+)
✅ Found HSD ID using pattern 1: 14026454583
📋 Extracted HSD ID: 14026454583
DEBUG: Section 1 content length: 2465
DEBUG: Section 1 preview: ### 1.1 Basic Information
**ID:** 14026454583  
**Title:** [ASUS][GX651A][duo eDP] under dGPU mode, 2nd LCD can't switch on/off when eDP2_HPD assert.  
**Main Issue:** 2nd LCD cannot switch on/off via hotplug in dGPU mode; works in iGPU mode.

### 1.2 Detailed Problem Description and Context
**Description:**  
- In dGPU 


Creating assessment for: Content Extraction & Summary



Creating assessment for: Issue Classification



Creating assessment for: Attachment Verification


...


Creating assessment for: Overall Tool Performance


HTML(value='\n    <div style="background-color: #f8f9fa; \n                border: 2px solid #17a2b8; \n      …

Button(button_style='success', description='💾 Save Assessment (14026454583)', layout=Layout(height='50px', wid…

Output()

📊 Assessment results will be saved to: C:\Users\tsontena\Documents\sighting\accuracy_test\data\14026454583_assessment_results.json
📝 Complete the interactive assessment to generate the structured results file
✅ Assessment completed for HSD 14026454583

🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊
🎊 FLEXIBLE WORKFLOW COMPLETE!
🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊
⏱️ Total Duration: 0:01:08.972344
📊 SUMMARY:
   📋 Total HSD IDs processed: 1
   ✅ Successful assessments: 1
   ❌ Failed assessments: 0
   📝 Source: Single HSD ID

📁 FILES CREATED:
   ✅ 14026454583:
      📄 GNAI output: \runs\14026454583_run.txt
      📊 Assessment: C:\Users\tsontena\Documents\sighting\accuracy_test\data\14026454583_assessment_results.json

⭐ Please complete the accuracy assessment for each successful HSD ID

📋 FINAL RESULTS SUMMARY:
✅ 14026454583: SUCCESS
